# PMA Early Warnings - Recomendacion Agroclimática Estacional

**Proyecto:** Plataforma de Monitoreo Agroclimático (PMA)  
**Fuentes:** IDEAM (pronóstico estacional) + CHIRPS/AgERA5 (histórico)  
**Departamentos:** Caquetá - Amazonas - Putumayo

Produce para el mes objetivo:
1. Mapa de cambio porcentual de precipitación respecto a la climatología 1982-2025.
2. Mapa de categorías SPEI (Muy seco / Normal / Muy húmedo) mediante años análogos.
3. GeoTIFFs de ambas capas, listos para consumo externo.

In [ ]:
# Instalacion de dependencias (solo en Google Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q rioxarray rasterio geopandas netCDF4 xarray requests matplotlib numpy
    print('Dependencias instaladas.')
else:
    print('Entorno local detectado.')

## 1. Parametros del flujo de trabajo

**Modifica aqui** el mes y anio objetivo, y los umbrales de QC.
El resto del notebook se ejecuta automaticamente a partir de estos valores.

In [ ]:
# ============================================================
# PARAMETROS - MODIFICA AQUI
# ============================================================

# Periodo objetivo
MONTH = 4          # mes objetivo  (1-12)
YEAR  = 2026       # anio objetivo

# URL base pronostico IDEAM
# Plantilla: ENSAMBLE_PREC_MENSUAL_{MM}_{YYYY}.nc
IDEAM_BASE_URL = "https://bart.ideam.gov.co/wrfideam/new_modelo/CPT/netcdf/PREC"

# Archivo historico de precipitacion y SPEI
# En Colab se descarga desde GitHub; en local apunta a tu copia.
HIST_GITHUB_URL = (
    "https://raw.githubusercontent.com/dagudelo30/PMA-early_warnings/main/"
    "data/precip_spei_mensual.nc"
)
HIST_LOCAL_PATH = "../data/precip_spei_mensual.nc"

# Departamentos a procesar
DEPARTMENTS = ["Caquet\u00e1", "Amazonas", "Putumayo"]
DEPT_NE_NAMES = {
    "Caquet\u00e1":  "Caquet\u00e1",
    "Amazonas": "Amazonas",
    "Putumayo": "Putumayo",
}

# Control de calidad (QC)
QC_UPPER_FACTOR = 2.0   # si pred > factor * clim -> se trunca
QC_CAP_FACTOR   = 1.1   # cap: pred truncada a cap * clim

# Parametro K (top-K analogos para moda de SPEI)
K = 3                   # numero de anios analogos

# Directorio de salida
OUTPUT_DIR = "../outputs"

# ============================================================
# No modificar debajo de esta linea
# ============================================================
import os
MONTH_STR = f"{MONTH:02d}"
PERIOD_TAG = f"{MONTH_STR}_{YEAR}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Periodo objetivo: {MONTH_STR}/{YEAR}")
print(f"Directorio de salida: {os.path.abspath(OUTPUT_DIR)}")

## 2. Importar librerias

In [ ]:
import sys, os, warnings
import requests
import numpy as np
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap, Normalize
from netCDF4 import Dataset
from xarray.backends import NetCDF4DataStore
from rasterio.enums import Resampling

warnings.filterwarnings('ignore')
print('Librerias importadas correctamente.')

## 3. Cargar pronostico IDEAM

Se descarga en memoria el ensamble mensual de precipitacion del IDEAM para el periodo objetivo.

In [ ]:
ideam_url = f"{IDEAM_BASE_URL}/ENSAMBLE_PREC_MENSUAL_{MONTH_STR}_{YEAR}.nc"
print(f"Descargando pronostico IDEAM: {ideam_url}")

r = requests.get(ideam_url, timeout=120)
r.raise_for_status()

nc  = Dataset("inmemory.nc", memory=r.content)
ds  = xr.open_dataset(NetCDF4DataStore(nc)).rename({"latitude": "y", "longitude": "x"})
ds  = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds  = ds.rio.write_crs("EPSG:4326", inplace=False)

print(f"Pronostico cargado. Variables: {list(ds.data_vars)}")
print(f"Extension: x=[{float(ds.x.min()):.2f}, {float(ds.x.max()):.2f}]  y=[{float(ds.y.min()):.2f}, {float(ds.y.max()):.2f}]")
ds

## 4. Cargar datos historicos (CHIRPS / AgERA5)

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB or not os.path.exists(HIST_LOCAL_PATH):
    print("Descargando historico desde GitHub...")
    r2 = requests.get(HIST_GITHUB_URL, timeout=300)
    r2.raise_for_status()
    nc2 = Dataset("hist_inmemory.nc", memory=r2.content)
    hist = xr.open_dataset(NetCDF4DataStore(nc2)).transpose("time", "y", "x")
else:
    print(f"Cargando historico local: {HIST_LOCAL_PATH}")
    hist = xr.open_dataset(HIST_LOCAL_PATH).transpose("time", "y", "x")

ds_filter = hist.sel(time=hist["time"].dt.month == MONTH)
print(f"Historico cargado. Variables: {list(ds_filter.data_vars)}")
print(f"Anios disponibles: {len(ds_filter.time)}")
ds_filter

## 5. Alinear grillas (reproyectar historico a grilla IDEAM)

In [ ]:
lat_min = float(ds_filter["y"].min())
lat_max = float(ds_filter["y"].max())
lon_min = float(ds_filter["x"].min())
lon_max = float(ds_filter["x"].max())

lat = ds["y"]
if lat[0] < lat[-1]:
    ds_clip = ds.sel(y=slice(lat_min, lat_max), x=slice(lon_min, lon_max))
else:
    ds_clip = ds.sel(y=slice(lat_max, lat_min), x=slice(lon_min, lon_max))

precip = ds_filter["precip"].rio.reproject_match(ds_clip, resampling=Resampling.bilinear)
spei   = ds_filter["spei"].rio.reproject_match(ds_clip, resampling=Resampling.nearest).round()

print(f"Grillas alineadas. Pronostico: {dict(ds_clip.dims)}  Historico: {dict(precip.dims)}")

## 6. Control de calidad (QC) al pronostico IDEAM

Dos reglas por pixel:
- Valores **negativos** -> reemplazados por 0 mm.
- Valores **> QC_UPPER_FACTOR x climatologia** -> truncados a QC_CAP_FACTOR x climatologia.

In [ ]:
clim = precip.mean("time", skipna=True)
pred = ds_clip["Prediccion(mm)"].clip(min=0)
pred_qc = xr.where(pred > QC_UPPER_FACTOR * clim, QC_CAP_FACTOR * clim, pred)
ds_clip["Prediccion_QC(mm)"] = pred_qc

changed   = (pred_qc != pred) & pred.notnull()
n_changed = int(changed.sum())
n_total   = int(pred.notnull().sum())
print(f"QC completado: {n_changed} / {n_total} pixeles modificados ({100*n_changed/n_total:.2f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
changed.astype('int8').plot(ax=ax, cmap='Reds', add_colorbar=False)
ax.set_title(f'Pixeles modificados por QC ({MONTH_STR}/{YEAR})')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/qc_pixels_{PERIOD_TAG}.png", dpi=150)
plt.show()

## 7. Cambio porcentual de precipitacion

Delta(%) = 100 * (Pf - Pclim) / Pclim

In [ ]:
eps = 1e-6
cambio_pct = xr.where(clim > eps, (pred_qc - clim) / clim * 100, 0)
chg_hist   = xr.where(clim > eps, (precip - clim)  / clim * 100, 0)
print(f"Cambio porcentual: [{float(cambio_pct.min()):.1f}%, {float(cambio_pct.max()):.1f}%]")

## 8. Anios analogos y moda SPEI (Top-K)

Para cada pixel:
1. Distancia absoluta entre Delta_hist(t) y Delta_f.
2. Se seleccionan los K tiempos historicos mas cercanos.
3. La recomendacion es la moda del SPEI en esos K analogos.

In [ ]:
print(f"Calculando Top-{K} analogos por pixel...")

diff  = xr.apply_ufunc(np.abs, chg_hist - cambio_pct).transpose("time", "y", "x")
spei2 = spei.transpose("time", "y", "x")

d        = diff.values
T, Y, X  = d.shape
N        = Y * X
d2       = d.reshape(T, N)

idx  = np.argpartition(d2, K-1, axis=0)[:K, :]
dK   = np.take_along_axis(d2, idx, axis=0)
ordK = np.argsort(dK, axis=0)
idxK = np.take_along_axis(idx, ordK, axis=0)

spei_np   = spei2.values.reshape(T, N)
spei_topK = spei_np[idxK, np.arange(N)]

mode   = np.full(N, np.nan, dtype=np.float32)
p_mode = np.full(N, np.nan, dtype=np.float32)

for j in range(N):
    a = spei_topK[:, j]
    a = a[np.isfinite(a)]
    if a.size == 0:
        continue
    a = a.astype(np.int32)
    vals, cnts = np.unique(a, return_counts=True)
    m        = vals[np.argmax(cnts)]
    mode[j]  = m
    p_mode[j] = (a == m).mean()

spei_mode = xr.DataArray(
    mode.reshape(Y, X).astype("int16"),
    dims=("y", "x"),
    coords={"y": diff["y"], "x": diff["x"]},
    name="spei_moda"
)
spei_mode = spei_mode.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
spei_mode = spei_mode.rio.write_crs("EPSG:4326", inplace=False)

print(f"Analogos calculados (K={K}).")
cats, cnt = np.unique(mode[np.isfinite(mode)].astype(int), return_counts=True)
label_map = {1: 'Muy seco', 2: 'Normal', 3: 'Muy humedo'}
for c, n in zip(cats, cnt):
    print(f"  Cat {c} ({label_map.get(c,'?')}): {n} pixeles ({100*n/N:.1f}%)")

## 9. Cargar limites departamentales (Natural Earth)

In [ ]:
print("Descargando limites administrativos Colombia (Natural Earth 10m)...")
url_adm1 = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip"
adm1     = gpd.read_file(url_adm1)
col_adm1 = adm1[adm1["admin"].str.lower() == "colombia"].copy()

depts = {}
for dept in DEPARTMENTS:
    ne_name = DEPT_NE_NAMES[dept]
    gdf     = col_adm1[col_adm1["name"] == ne_name]
    if gdf.empty:
        gdf = col_adm1[col_adm1["name"].str.contains(ne_name, case=False, na=False)]
    depts[dept] = gdf
    print(f"  {dept}: {len(gdf)} poligono(s)")

print("Limites cargados.")

## 10. Funciones de visualizacion

In [ ]:
def add_north_arrow(ax, lon, lat, size_deg=0.5):
    ax.annotate(
        '', xy=(lon, lat + size_deg), xytext=(lon, lat),
        arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
        zorder=10
    )
    ax.text(lon, lat + size_deg * 1.3, 'N', ha='center', va='bottom',
            fontsize=9, fontweight='bold', zorder=10)


def add_scalebar(ax, lon_start, lat_bar, length_km=200, height_deg=0.1):
    deg_per_km = 1 / (111 * abs(np.cos(np.radians(lat_bar))) + 1e-9)
    length_deg = length_km * deg_per_km
    lon_end    = lon_start + length_deg
    y0, y1     = lat_bar - height_deg / 2, lat_bar + height_deg / 2
    ax.add_patch(plt.Rectangle((lon_start, y0), length_deg/2, height_deg,
                                fc='black', ec='black', zorder=10))
    ax.add_patch(plt.Rectangle((lon_start + length_deg/2, y0), length_deg/2, height_deg,
                                fc='white', ec='black', zorder=10))
    ax.text(lon_start, y0 - height_deg*0.6, '0', ha='center', fontsize=7, zorder=10)
    ax.text(lon_end,   y0 - height_deg*0.6, f'{length_km} km', ha='center', fontsize=7, zorder=10)


print("Funciones de visualizacion definidas.")

## 11. Generar mapas y GeoTIFFs por departamento

In [ ]:
cmap_div = LinearSegmentedColormap.from_list(
    "cambio_pct",
    [(0.00, "#db1b1b"), (0.25, "#f97c2b"),
     (0.50, "#ecfc99"), (0.75, "#2fb47c"), (1.00, "#1ea0c4")],
    N=256
)

SPEI_LABELS = {1: "Verano\n(Muy seco)", 2: "Normal", 3: "Invierno\n(Muy humedo)"}
SPEI_COLORS = ["#d73027", "#ffffbf", "#1a9850"]
cmap_spei   = ListedColormap(SPEI_COLORS)
norm_spei   = BoundaryNorm([0.5, 1.5, 2.5, 3.5], cmap_spei.N)
VMIN, VMAX  = -100, 100

for dept in DEPARTMENTS:
    gdf = depts[dept]
    if gdf.empty:
        print(f"Sin geometria para {dept}, se omite.")
        continue

    print(f"\nProcesando {dept}...")
    geom = gdf.geometry
    crs  = gdf.crs
    slug = dept.lower().replace('\u00e1','a').replace('\u00e9','e').replace('\u00fa','u').replace('\u00f3','o')

    da_chg = cambio_pct.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    da_chg = da_chg.rio.write_crs("EPSG:4326", inplace=False)
    da_chg_clip = da_chg.rio.clip(geom, crs, drop=True, all_touched=True)
    da_spei     = spei_mode.rio.clip(geom, crs, drop=True, all_touched=True)

    tif_chg  = f"{OUTPUT_DIR}/cambio_pct_{PERIOD_TAG}_{slug}.tif"
    tif_spei = f"{OUTPUT_DIR}/spei_moda_{PERIOD_TAG}_{slug}.tif"

    da_tmp = da_chg_clip
    if da_tmp.y.values[0] < da_tmp.y.values[-1]:
        da_tmp = da_tmp.isel(y=slice(None, None, -1)).rio.write_transform(inplace=True)
    da_tmp.rio.to_raster(tif_chg, driver="GTiff", dtype="float32", compress="LZW", nodata=float("nan"))

    da_sp = da_spei
    if da_sp.y.values[0] < da_sp.y.values[-1]:
        da_sp = da_sp.isel(y=slice(None, None, -1)).rio.write_transform(inplace=True)
    da_sp.rio.to_raster(tif_spei, driver="GTiff", dtype="int16", compress="LZW", nodata=-9999)

    print(f"  GeoTIFFs: {tif_chg}")
    print(f"            {tif_spei}")

    # Mapa cambio porcentual
    vmax_raw = float(da_chg_clip.max())
    norm_div = Normalize(vmin=VMIN, vmax=VMAX)
    ticks_cb = np.arange(VMIN, min(VMAX, np.floor(vmax_raw/20)*20+20), 20).astype(int)

    fig, ax = plt.subplots(figsize=(9, 6))
    da_chg_clip.plot(
        ax=ax, cmap=cmap_div, norm=norm_div, add_colorbar=True,
        cbar_kwargs=dict(shrink=0.85, pad=0.02, extend="neither",
                         ticks=ticks_cb, label="Cambio porcentual (%)")
    )
    gdf.boundary.plot(ax=ax, linewidth=2.2, color="white", zorder=5)
    gdf.boundary.plot(ax=ax, linewidth=1.1, color="0.15", zorder=6)
    xlim = ax.get_xlim(); ylim = ax.get_ylim()
    add_scalebar(ax, xlim[0]+0.05*(xlim[1]-xlim[0]), ylim[0]+0.08*(ylim[1]-ylim[0]))
    add_north_arrow(ax, xlim[0]+0.06*(xlim[1]-xlim[0]), ylim[0]+0.18*(ylim[1]-ylim[0]), size_deg=0.3)
    ax.set_title(f"Cambio porcentual {dept} - {MONTH_STR}/{YEAR}")
    ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
    ax.text(0.01, 0.01, "Fuente: (IDEAM, CHIRPS). Limite: Natural Earth 10m Admin1.",
            transform=ax.transAxes, fontsize=7.5, va='bottom')
    plt.tight_layout()
    png_chg = f"{OUTPUT_DIR}/mapa_cambio_pct_{PERIOD_TAG}_{slug}.png"
    plt.savefig(png_chg, dpi=400, bbox_inches='tight')
    plt.show()

    # Mapa SPEI
    fig2, ax2 = plt.subplots(figsize=(9, 6))
    im2 = da_spei.plot(
        ax=ax2, cmap=cmap_spei, norm=norm_spei, add_colorbar=True,
        cbar_kwargs=dict(shrink=0.85, pad=0.02, ticks=[1, 2, 3])
    )
    im2.colorbar.set_ticklabels([SPEI_LABELS[i] for i in [1, 2, 3]])
    gdf.boundary.plot(ax=ax2, linewidth=2.2, color="white", zorder=5)
    gdf.boundary.plot(ax=ax2, linewidth=1.1, color="0.15", zorder=6)
    xlim2 = ax2.get_xlim(); ylim2 = ax2.get_ylim()
    add_scalebar(ax2, xlim2[0]+0.05*(xlim2[1]-xlim2[0]), ylim2[0]+0.08*(ylim2[1]-ylim2[0]))
    add_north_arrow(ax2, xlim2[0]+0.06*(xlim2[1]-xlim2[0]), ylim2[0]+0.18*(ylim2[1]-ylim2[0]), size_deg=0.3)
    ax2.set_title(f"SPEI {dept} - {MONTH_STR}/{YEAR}")
    ax2.set_xlabel("Longitud"); ax2.set_ylabel("Latitud")
    ax2.text(0.01, 0.01, "Fuente: SPEI (CHIRPS, AgERA5). Limite: Natural Earth 10m Admin1.",
             transform=ax2.transAxes, fontsize=7.5, va='bottom')
    plt.tight_layout()
    png_spei = f"{OUTPUT_DIR}/mapa_spei_{PERIOD_TAG}_{slug}.png"
    plt.savefig(png_spei, dpi=400, bbox_inches='tight')
    plt.show()
    print(f"  Mapas guardados: {png_chg} | {png_spei}")

print("\nTodos los departamentos procesados.")

## 12. Resumen de archivos generados

In [ ]:
import glob
files = sorted(glob.glob(f"{OUTPUT_DIR}/*{PERIOD_TAG}*"))
print(f"Archivos generados para {PERIOD_TAG}:")
for f in files:
    size = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):55s}  {size:8.1f} KB")
print(f"\nTotal: {len(files)} archivo(s)")